# 🎵 Notebook 04 — Audio Preprocessing → Mel Spectrograms
Person 2

In [1]:
!pip install librosa soundfile tqdm matplotlib -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, librosa, numpy as np, soundfile as sf
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from PIL import Image

BASE_DIR   = "/content/drive/MyDrive/Colab Notebooks/deepfake-project"
IN_AUDIO   = os.path.join(BASE_DIR, "data/audio")
OUT_SPEC   = os.path.join(BASE_DIR, "data/spectrograms")
OUT_NPY    = os.path.join(BASE_DIR, "data/spectrograms_npy")

SR, N_MELS, N_FFT, HOP_LENGTH = 16000, 128, 1024, 512
DURATION    = 3.0
SPEC_SIZE   = 128

for d in [OUT_SPEC, OUT_NPY]:
    os.makedirs(d, exist_ok=True)
print("✅ Config ready.")

Mounted at /content/drive
✅ Config ready.


In [3]:
# ✅ Mel Spectrogram Function
def wav_to_melspec(wav_path, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                   hop=HOP_LENGTH, duration=DURATION, target_size=SPEC_SIZE):
    try:
        y, _ = librosa.load(wav_path, sr=sr, mono=True)
    except Exception as e:
        print(f"  [WARN] {wav_path}: {e}"); return []
    samples_per_seg = int(sr * duration)
    specs = []
    for start in range(0, len(y), samples_per_seg):
        segment = y[start : start + samples_per_seg]
        if len(segment) < samples_per_seg // 2: break
        if len(segment) < samples_per_seg:
            segment = np.pad(segment, (0, samples_per_seg - len(segment)))
        mel     = librosa.feature.melspectrogram(y=segment, sr=sr, n_fft=n_fft,
                                                  hop_length=hop, n_mels=n_mels)
        log_mel = librosa.power_to_db(mel, ref=np.max)
        log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
        img     = Image.fromarray((log_mel * 255).astype(np.uint8))
        img     = img.resize((target_size, target_size))
        specs.append(np.array(img, dtype=np.float32) / 255.0)
    return specs

print("✅ wav_to_melspec() defined.")

✅ wav_to_melspec() defined.


In [4]:
# ✅ Process all audio splits → PNG + NPY
def process_audio_split(split):
    in_sub   = os.path.join(IN_AUDIO, split)
    out_png  = os.path.join(OUT_SPEC, split)
    out_npy  = os.path.join(OUT_NPY,  split)
    os.makedirs(out_png, exist_ok=True)
    os.makedirs(out_npy, exist_ok=True)
    wavs = list(Path(in_sub).rglob("*.wav"))
    if not wavs: print(f"[SKIP] No wavs in {in_sub}"); return
    print(f"\n=== {split}: {len(wavs)} wav files ===")
    for wp in tqdm(wavs):
        specs = wav_to_melspec(str(wp))
        for idx, spec in enumerate(specs):
            name = f"{wp.stem}_seg{idx:03d}"
            # Save PNG
            fig, ax = plt.subplots(figsize=(1.28, 1.28), dpi=100)
            ax.imshow(spec, aspect="auto", origin="lower", cmap="magma")
            ax.axis("off"); plt.tight_layout(pad=0)
            plt.savefig(os.path.join(out_png, name+".png"), bbox_inches="tight", pad_inches=0)
            plt.close(fig)
            # Save NPY
            np.save(os.path.join(out_npy, name+".npy"), spec)
    total_npy = sum(1 for _ in Path(out_npy).glob("*.npy"))
    print(f"  ✅ {total_npy} spectrogram arrays saved")

for split in ["fake_with_audio", "real_with_audio"]:
    process_audio_split(split)
print("\n✅ Audio preprocessing complete.")


=== fake_with_audio: 100 wav files ===


100%|██████████| 100/100 [01:20<00:00,  1.25it/s]


  ✅ 499 spectrogram arrays saved

=== real_with_audio: 100 wav files ===


100%|██████████| 100/100 [00:54<00:00,  1.84it/s]

  ✅ 543 spectrogram arrays saved

✅ Audio preprocessing complete.


In [5]:
# ✅ Preview a few spectrograms
import random
def preview_spectrograms(split="real_with_audio", n=4):
    d  = os.path.join(OUT_SPEC, split)
    ps = list(Path(d).glob("*.png"))
    if not ps: print("None found."); return
    ps = random.sample(ps, min(n, len(ps)))
    fig, axes = plt.subplots(1, len(ps), figsize=(3*len(ps), 3))
    if len(ps) == 1: axes = [axes]
    for ax, p in zip(axes, ps):
        img = plt.imread(str(p))
        ax.imshow(img, cmap="magma"); ax.axis("off"); ax.set_title(p.stem[:15])
    plt.suptitle(f"Mel Spectrograms — {split}"); plt.tight_layout(); plt.show()

preview_spectrograms("real_with_audio")
preview_spectrograms("fake_with_audio")